In [4]:
import os
import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, BatchNormalization,
                                     Flatten, Dense, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configuración
IMG_SIZE = 128
CHANNELS = 3
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_IMG_DIR = "data/train/train"
TEST_IMG_DIR = "data/test/test"
TRAIN_CSV = "data/train.csv"           
TEST_CSV = "data/test.csv"  

def load_images(df, img_dir):
    """Carga imágenes con manejo de errores y balance de canales"""
    images = []
    valid_ids = []
    
    for idx in df['id']:
        file_path = os.path.join(img_dir, f"clips-{idx}.png")
        try:
            img = cv2.imread(file_path, cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Verificar canales faltantes
            if img.shape[2] != CHANNELS:
                img = np.stack([img]*CHANNELS, axis=-1)[:,:,:CHANNELS]
                
            images.append(img)
            valid_ids.append(idx)
        except Exception as e:
            print(f"Error cargando {file_path}: {str(e)}")
    
    return np.array(images), valid_ids

# Carga de datos mejorada
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

X_train, train_ids = load_images(train_df, TRAIN_IMG_DIR)
y_train = train_df[train_df['id'].isin(train_ids)]['clip_count'].values

X_test, test_ids = load_images(test_df, TEST_IMG_DIR)
test_df = test_df[test_df['id'].isin(test_ids)]

# Normalización
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Data Augmentation
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Modelo mejorado
def create_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Dropout(0.2),
        
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Dropout(0.3),
        
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Dropout(0.4),
        
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(1)
    ])
    
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer,
                  loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(),
                           tf.keras.metrics.MeanAbsoluteError()])
    return model

# Callbacks
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.1, patience=3)
]

# Entrenamiento con aumento de datos
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

model = create_model()
history = model.fit(
    train_datagen.flow(X_tr, y_tr, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    steps_per_epoch=len(X_tr)//BATCH_SIZE
)

# Evaluación
val_loss, val_rmse, val_mae = model.evaluate(X_val, y_val)
print(f"Validation RMSE: {val_rmse:.4f}, MAE: {val_mae:.4f}")

# Predicción
predictions = model.predict(X_test).flatten()

# Submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'clip_count': predictions
})
submission.to_csv('submission9.csv', index=False)

Epoch 1/10


c:\Users\borja\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\borja\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


375/375 ━━━━━━━━━━━━━━━━━━━━ 129s 339ms/step - loss: 1156.0742 - mean_absolute_error: 31.9858 - root_mean_squared_error: 33.7280 - val_loss: 176.4892 - val_mean_absolute_error: 11.1483 - val_root_mean_squared_error: 13.2849 - learning_rate: 0.0010
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 132s 351ms/step - loss: 51.4758 - mean_absolute_error: 5.6285 - root_mean_squared_error: 7.1604 - val_loss: 2257.6079 - val_mean_absolute_error: 44.4159 - val_root_mean_squared_error: 47.5143 - learning_rate: 0.0010
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 135s 361ms/step - loss: 40.9732 - mean_absolute_error: 5.0334 - root_mean_squared_error: 6.3998 - val_loss: 71.5338 - val_mean_absolute_error: 7.2160 - val_root_mean_squared_error: 8.4578 - learning_rate: 0.0010
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 138s 368ms/step - loss: 37.4297 - mean_absolute_error: 4.8420 - root_mean_squared_error: 6.1165 - val_loss: 85.0819 - val_mean_absolute_error: 8.6387 - val_root_mean_squared_error: 9.2240 - learning_rate: